In [1]:
from kbel.disambiguators import Disambiguator
from kbel.disambiguators.naive import NaiveDisambiguator
from kbel.disambiguators.similarity import SimilarityDisambiguator
from kbel.core.mention import Mention
from kbel.core.mention import EntityType
from kbel.knowledge_sources import KnowledgeSource
from kif_lib import Search
# import logging
# logging.basicConfig(level=logging.DEBUG)

/Users/marcelomachado/Documents/projects/kbel/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Using `naive` disambiguator to link entities from Wikidata

In [5]:
kbel = Disambiguator(strategy_name='naive')
results = kbel.disambiguate(
    mention=Mention(label='rock', text='', entity_type=EntityType.ITEM),
    ks=KnowledgeSource('wikidata', limit=10))
display (*results)

('rock music',
 'popular music genre',
 Item(IRI('http://www.wikidata.org/entity/Q11399')))

### Using `similarity` disambiguator to link entities from Wikidata

In [6]:
kbel = Disambiguator(strategy_name='sim')
results = kbel.disambiguate(
    ks=KnowledgeSource('wikidata-wapi', limit=10),
    mention=Mention(label='Rock', text='Rock is a stone', entity_type=EntityType.ITEM),
    limit=1)
display (*results)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 33596.18it/s]


('stone',
 'rock; building material',
 Item(IRI('http://www.wikidata.org/entity/Q22731')))

### Using `LLM` disambiguator to link entities from Wikidata

Instantiating LLM Disambiguator with IBM WatsonX's models

In [ ]:
import os
import dotenv
dotenv.load_dotenv()

kbel = Disambiguator(
    'llm',
    model_name='meta-llama/llama-3-3-70b-instruct',
    model_provider='ibm',
    model_apikey=os.environ['LLM_API_KEY'],
    model_endpoint=os.environ['LLM_API_ENDPOINT'],
)

In [ ]:
results = kbel.disambiguate_item(
    label='Rock',
    searcher=Search('wikidata-wapi', limit=100),
    sentence='A rock can be used in construction to mimic the appearance and durability of natural stone.')

display (*results)

('rock',
 'mass of stone projecting out of the ground or water',
 Item(IRI('http://www.wikidata.org/entity/Q1404150')))

('rock',
 'naturally occurring solid aggregate of one or more minerals or mineraloids',
 Item(IRI('http://www.wikidata.org/entity/Q8063')))

LLM disambiguator uses [LangChain ChatModels](https://python.langchain.com/docs/integrations/providers/). Below, we use IBM WatsonX's models

In [ ]:
from langchain_ibm import ChatWatsonx
model = ChatWatsonx(
    model_id='meta-llama/llama-3-3-70b-instruct',
    apikey=os.environ['LLM_API_KEY'], # type: ignore
    url=os.environ['LLM_API_ENDPOINT'], # type: ignore
    project_id=os.environ['WATSONX_PROJECT_ID'],
    temperature=0.0
)
kbel = Disambiguator(disambiguator_name='llm', model=model)

In [ ]:
results = kbel.disambiguate_item(
    label='Rock',
    searcher=Search('wikidata-wapi', limit=100),
    sentence='A rock can be used in construction to mimic the appearance and durability of natural stone.')
display (*results)

('rock',
 'mass of stone projecting out of the ground or water',
 Item(IRI('http://www.wikidata.org/entity/Q1404150')))

('rock',
 'naturally occurring solid aggregate of one or more minerals or mineraloids',
 Item(IRI('http://www.wikidata.org/entity/Q8063')))

Disambiguating properties:

In [ ]:
results = kbel.disambiguate_property(
    label='instance of', searcher=Search('wikidata-wapi', limit=20), sentence='John is a Human')
display(*results)

('instance of',
 'type to which this subject corresponds/belongs. Different from P279 (subclass of); for example: K2 is an instance of mountain; volcanoes form a subclass of mountains',
 Property(IRI('http://www.wikidata.org/entity/P31'), None))

Linking entities to DBpedia

In [ ]:
searcher = Search('dbpedia', limit=100)
results = kbel.disambiguate_item(label='Rock', searcher=searcher, sentence='A rock can be used in construction to mimic the appearance and durability of natural stone.')
display(*results)

('Metamorphic rock',
 '',
 Item(IRI('http://dbpedia.org/resource/Metamorphic_rock')))

('Sedimentary rock',
 '',
 Item(IRI('http://dbpedia.org/resource/Sedimentary_rock')))

Linking entities to PubChem

In [ ]:
results = kbel.disambiguate_item(label='benzene', searcher=Search('pubchem-ddgs', limit=20), sentence='A benzene can be used in construction to mimic the appearance and durability of natural stone.')
display(*results)

('PubChem Benzene | C6H6 | CID 241 - PubChem',
 'Benzene | C6H6 | CID 241 - structure, chemical names, physical and chemical properties, classification, patents, literature, biological activities, safety/hazards/toxicity information, supplier lists, and more.',
 Item(IRI('http://rdf.ncbi.nlm.nih.gov/pubchem/compound/CID241')))